# Deepert time-lapse ERT regularization benchmark

这个 notebook 用来快速测试 Deepert 时移 ERT 反演里的不同组合：

- 数据目标函数：`weighted_log_l2`, `weighted_log_l1`, `weighted_log_huber`, `log_data_difference_l2`
- 空间正则化：`first_order_smoothness`, `structural_prior`, `model_difference_smoothness`, `spatial_total_variation`, `spatial_huber`
- 时间正则化：`temporal_smoothness`, `second_order_temporal_smoothness`, `temporal_total_variation`, `temporal_huber`, `active_time_constraint`, `baseline_reference`
- 优化算法：默认 `gauss_newton_cgls`，也可以把公共参数里的 `OPTIMIZER` 改成 `levenberg_marquardt`, `lbfgs`, `lbfgs_b`, `nonlinear_cg`, `adam`

默认是**快测模式**：只跑前 `MAX_TIMESTEPS=30` 个时间步，`MAX_ITERATIONS=6`，并且加 `--no-plot --quiet`，用来先筛选值得跑全年 365 天的组合。

推荐先看：

1. 哪些组合能稳定收敛。
2. 哪些组合耗时明显变慢。
3. 哪些组合保留边界/事件响应更合理。
4. 最后再把 1-2 个最有希望的组合跑全年。


In [ ]:
from __future__ import annotations

import json
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import pandas as pd
except Exception:
    pd = None

# Robust repo-root detection. Works whether the notebook kernel starts in repo root or examples/.
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'deepert').exists() else cwd.parent
SCRIPT = ROOT / 'examples' / '2_timelapsedERT_inversion_deepert.py'
FORWARD_DIR = ROOT / 'result' / '1_timelapsedERT_forward_deepert'
TRUE_MODEL_DIR = ROOT / 'resistivity_models_2d'
BENCH_ROOT = ROOT / 'result' / '3_regularization_benchmark_deepert'
BENCH_ROOT.mkdir(parents=True, exist_ok=True)

print('ROOT       =', ROOT)
print('SCRIPT     =', SCRIPT)
print('FORWARD_DIR=', FORWARD_DIR)
print('BENCH_ROOT =', BENCH_ROOT)

## Quick-test controls

如果只是确认代码是否能跑，建议：

```python
MAX_TIMESTEPS = 12
MAX_ITERATIONS = 3
```

如果要比较结果形态，建议：

```python
MAX_TIMESTEPS = 30 或 60
MAX_ITERATIONS = 6 或 8
```

如果要最终论文结果，再改成：

```python
MAX_TIMESTEPS = None
MAX_ITERATIONS = 15
```

In [ ]:
# Fast default benchmark settings.
MAX_TIMESTEPS = 30       # None means all timesteps; keep small for quick screening.
MAX_ITERATIONS = 6       # Full production default is 15.
CGLS_MAX_ITERATIONS = 40 # Full production default is 60.
WINDOW_SIZE = 3
WINDOW_STEP = 1
FILE_STRIDE = 1
FORCE_RERUN = False      # Set True to overwrite existing benchmark outputs.

# OPTIMIZER is the outer nonlinear update; LINEARIZED_SOLVER is used by Gauss-Newton/LM.
OPTIMIZER = 'gauss_newton_cgls'
LINEARIZED_SOLVER = 'gpu_cgls'

COMMON_ARGS = [
    '--forward-dir', str(FORWARD_DIR),
    '--true-model-dir', str(TRUE_MODEL_DIR),
    '--inversion-mode', 'windowed',
    '--window-size', str(WINDOW_SIZE),
    '--window-step', str(WINDOW_STEP),
    '--file-stride', str(FILE_STRIDE),
    '--max-iterations', str(MAX_ITERATIONS),
    '--cgls-max-iterations', str(CGLS_MAX_ITERATIONS),
    '--optimizer', OPTIMIZER,
    '--linearized-solver', LINEARIZED_SOLVER,
    '--no-plot',
    '--quiet',
]
if MAX_TIMESTEPS is not None:
    COMMON_ARGS += ['--max-timesteps', str(MAX_TIMESTEPS)]

print('Common args:')
print(' '.join(COMMON_ARGS))


## Test cases

默认组合建议如下：

- `weighted_log_l2_baseline`: 当前默认，作为速度和结果基准。
- `log_data_difference_l2`: 时移数据差目标函数，对比绝对数据拟合和变化量拟合。
- `structural_prior`: 结构先验空间光滑，跨地层边界弱约束。
- `model_difference_smoothness`: 约束时移变化量的空间平滑。
- `active_time_constraint`: 主动时间约束，允许事件期变化更灵活。
- `baseline_reference`: 基准参考/交叉模型约束。
- `second_order_temporal_smoothness`: 时间二阶平滑，看季节趋势是否更自然。
- `spatial_total_variation` / `temporal_total_variation`: 边界/突变保留更强，但可能更慢、更块状。
- `weighted_log_huber`: 鲁棒数据目标函数，看抗异常数据效果。

可以先少跑几组，比如只保留 `weighted_log_l2_baseline`, `structural_prior`, `active_time_constraint`, `weighted_log_huber`。


In [ ]:
TEST_CASES = [
    {
        'name': 'weighted_log_l2_baseline',
        'label': 'Weighted log L2 baseline',
        'args': [
            '--data-misfit', 'weighted_log_l2',
            '--spatial-regularization', 'first_order_smoothness',
            '--temporal-regularization-type', 'temporal_smoothness',
        ],
    },
    {
        'name': 'log_data_difference_l2',
        'label': 'Log data-difference L2',
        'args': [
            '--data-misfit', 'log_data_difference_l2',
            '--spatial-regularization', 'first_order_smoothness',
            '--temporal-regularization-type', 'temporal_smoothness',
        ],
    },
    {
        'name': 'structural_prior',
        'label': 'Structural-prior smoothness',
        'args': [
            '--data-misfit', 'weighted_log_l2',
            '--spatial-regularization', 'structural_prior',
            '--temporal-regularization-type', 'temporal_smoothness',
        ],
    },
    {
        'name': 'model_difference_smoothness',
        'label': 'Model-difference smoothness',
        'args': [
            '--data-misfit', 'weighted_log_l2',
            '--spatial-regularization', 'model_difference_smoothness',
            '--temporal-regularization-type', 'temporal_smoothness',
        ],
    },
    {
        'name': 'active_time_constraint',
        'label': 'Active time constraint (ATC)',
        'args': [
            '--data-misfit', 'weighted_log_l2',
            '--spatial-regularization', 'first_order_smoothness',
            '--temporal-regularization-type', 'active_time_constraint',
        ],
    },
    {
        'name': 'baseline_reference',
        'label': 'Baseline-reference constraint',
        'args': [
            '--data-misfit', 'weighted_log_l2',
            '--spatial-regularization', 'first_order_smoothness',
            '--temporal-regularization-type', 'baseline_reference',
        ],
    },
    {
        'name': 'second_order_temporal_smoothness',
        'label': 'Second-order temporal smoothness',
        'args': [
            '--data-misfit', 'weighted_log_l2',
            '--spatial-regularization', 'first_order_smoothness',
            '--temporal-regularization-type', 'second_order_temporal_smoothness',
        ],
    },
    {
        'name': 'spatial_total_variation',
        'label': 'Spatial total variation',
        'args': [
            '--data-misfit', 'weighted_log_l2',
            '--spatial-regularization', 'spatial_total_variation',
            '--temporal-regularization-type', 'temporal_smoothness',
        ],
    },
    {
        'name': 'temporal_total_variation',
        'label': 'Temporal total variation',
        'args': [
            '--data-misfit', 'weighted_log_l2',
            '--spatial-regularization', 'first_order_smoothness',
            '--temporal-regularization-type', 'temporal_total_variation',
        ],
    },
    {
        'name': 'weighted_log_huber',
        'label': 'Weighted log Huber',
        'args': [
            '--data-misfit', 'weighted_log_huber',
            '--spatial-regularization', 'first_order_smoothness',
            '--temporal-regularization-type', 'temporal_smoothness',
        ],
    },
]

for case in TEST_CASES:
    print(case['name'], '->', ' '.join(case['args']))


## Run benchmark cases

这一格会实际运行反演。每个 case 输出到：

```text
result/3_regularization_benchmark_deepert/<case_name>/
```

如果已经跑过，并且 `FORCE_RERUN=False`，会直接读取已有 summary，不重复计算。

In [ ]:
def _expected_arg(case: dict, option: str, default=None):
    args = case.get('args', [])
    for idx, value in enumerate(args[:-1]):
        if value == option:
            return args[idx + 1]
    return default


def _cache_matches_case(summary: dict, case: dict) -> bool:
    expected = {
        'data_misfit': _expected_arg(case, '--data-misfit'),
        'spatial_regularization': _expected_arg(case, '--spatial-regularization'),
        'temporal_regularization_type': _expected_arg(case, '--temporal-regularization-type'),
        'optimizer': _expected_arg(case, '--optimizer', OPTIMIZER),
    }
    return all(expected[key] is None or summary.get(key) == expected[key] for key in expected)


def run_case(case: dict) -> dict:
    out_dir = BENCH_ROOT / case['name']
    summary_file = out_dir / 'timelapsed_inversion_summary.json'
    log_file = out_dir / 'run.log'
    out_dir.mkdir(parents=True, exist_ok=True)

    if summary_file.exists() and not FORCE_RERUN:
        summary = json.loads(summary_file.read_text())
        if _cache_matches_case(summary, case):
            summary['case_name'] = case['name']
            summary['case_label'] = case['label']
            summary['status'] = 'cached'
            return summary
        print(f"Cache mismatch for {case['name']}; rerunning with canonical arguments.")

    cmd = [
        'uv', 'run', 'python', str(SCRIPT),
        '--output-dir', str(out_dir),
        *COMMON_ARGS,
        *case['args'],
    ]
    print('\nRunning:', case['name'])
    print(' '.join(cmd))
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    elapsed = time.perf_counter() - t0
    log_file.write_text(
        'COMMAND:\n' + ' '.join(cmd) + '\n\nSTDOUT:\n' + proc.stdout + '\n\nSTDERR:\n' + proc.stderr,
        encoding='utf-8',
    )
    if proc.returncode != 0:
        raise RuntimeError(f"Case {case['name']} failed with code {proc.returncode}. See {log_file}")
    if not summary_file.exists():
        raise FileNotFoundError(f"Expected summary was not created: {summary_file}")

    summary = json.loads(summary_file.read_text())
    summary['case_name'] = case['name']
    summary['case_label'] = case['label']
    summary['status'] = 'ran'
    summary['notebook_elapsed_sec'] = elapsed
    return summary

summaries = []
for case in TEST_CASES:
    summaries.append(run_case(case))

print(f'Finished {len(summaries)} cases.')


## Summary table

这里汇总每组的耗时、窗口数、最终 chi2 等。快测时主要看相对差异即可。

In [ ]:
def load_chi2(out_dir: Path) -> np.ndarray:
    path = out_dir / 'chi2_all.npy'
    return np.load(path) if path.exists() else np.asarray([])

def load_window_reports(out_dir: Path) -> list[dict]:
    path = out_dir / 'window_reports.json'
    if path.exists():
        return json.loads(path.read_text())
    return []

rows = []
for s in summaries:
    out_dir = Path(s['output_dir'])
    chi2 = load_chi2(out_dir)
    wr = load_window_reports(out_dir)
    window_times = np.asarray([w.get('elapsed_sec', np.nan) for w in wr], dtype=float) if wr else np.asarray([])
    rows.append({
        'case': s['case_name'],
        'label': s['case_label'],
        'status': s.get('status'),
        'elapsed_min': s.get('elapsed_min', np.nan),
        'n_timesteps': s.get('n_timesteps'),
        'n_windows': s.get('n_windows'),
        'max_iterations': s.get('max_iterations'),
        'final_chi2': float(chi2[-1]) if chi2.size else np.nan,
        'mean_window_sec': float(np.nanmean(window_times)) if window_times.size else np.nan,
        'data_misfit': s.get('data_misfit'),
        'spatial': s.get('spatial_regularization'),
        'temporal': s.get('temporal_regularization_type'),
        'out_dir': str(out_dir),
    })

if pd is not None:
    df = pd.DataFrame(rows)
    display(df)
else:
    for row in rows:
        print(row)

## Speed and chi2 comparison

耗时越低越好；chi2 只是数据拟合指标，不能单独代表模型物理合理性。Robust 正则化通常会稍慢，关键看是否换来了更稳定/更可解释的模型变化。

In [ ]:
labels = [r['label'] for r in rows]
elapsed = np.asarray([r['elapsed_min'] for r in rows], dtype=float)
final_chi2 = np.asarray([r['final_chi2'] for r in rows], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
axes[0].barh(labels, elapsed, color='steelblue')
axes[0].set_xlabel('Elapsed time [min]')
axes[0].set_title('Runtime')
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(labels, final_chi2, color='darkorange')
axes[1].set_xlabel('Final chi2')
axes[1].set_title('Final data fit')
axes[1].grid(axis='x', alpha=0.3)
plt.show()

## Chi2 curves

这里画每个 case 的窗口 final chi2 序列。windowed 反演中，`chi2_all.npy` 是每个窗口结束后的 chi2。

In [ ]:
plt.figure(figsize=(11, 4.5))
for row in rows:
    out_dir = Path(row['out_dir'])
    chi2 = load_chi2(out_dir)
    if chi2.size:
        plt.plot(np.arange(1, chi2.size + 1), chi2, lw=1.8, label=row['label'])
plt.xlabel('Window index')
plt.ylabel('Final chi2 per window')
plt.grid(alpha=0.3)
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## Model snapshot comparison

下面用反演网格画某一个时间步的模型。默认画 benchmark 中间时刻。

注意：这里画的是 `log10 resistivity`，方便不同组合统一色标比较。

In [ ]:
def load_mesh(out_dir: Path):
    with np.load(out_dir / 'timelapse_inversion_mesh.npz') as data:
        return np.asarray(data['nodes']), np.asarray(data['cells'], dtype=np.int32)

def load_models(out_dir: Path):
    return np.load(out_dir / 'final_models.npy')

def load_steps(out_dir: Path):
    path = out_dir / 'steps.npy'
    return np.load(path) if path.exists() else None

available_rows = [r for r in rows if (Path(r['out_dir']) / 'final_models.npy').exists()]
if not available_rows:
    raise RuntimeError('No final_models.npy files found. Run benchmark cases first.')

# Pick the middle benchmark timestep.
ref_out = Path(available_rows[0]['out_dir'])
ref_models = load_models(ref_out)
plot_time_index = ref_models.shape[1] // 2
steps = load_steps(ref_out)
step_label = f"t={int(steps[plot_time_index])}" if steps is not None else f"index={plot_time_index}"

# Shared color range across cases at this selected time.
all_values = []
for row in available_rows:
    models = load_models(Path(row['out_dir']))
    all_values.append(np.log10(np.maximum(models[:, plot_time_index], np.finfo(float).tiny)))
all_values = np.concatenate(all_values)
vmin, vmax = np.nanpercentile(all_values, [2, 98])

n = len(available_rows)
fig, axes = plt.subplots(n, 1, figsize=(11, 2.1 * n), constrained_layout=True)
if n == 1:
    axes = [axes]

last_mappable = None
for ax, row in zip(axes, available_rows):
    out_dir = Path(row['out_dir'])
    nodes, cells = load_mesh(out_dir)
    models = load_models(out_dir)
    values = np.log10(np.maximum(models[:, plot_time_index], np.finfo(float).tiny))
    last_mappable = ax.tripcolor(nodes[:, 0], nodes[:, 1], cells, facecolors=values, shading='flat', cmap='turbo', vmin=vmin, vmax=vmax)
    ax.set_aspect('equal', adjustable='box')
    ax.set_ylabel('Z [m]')
    ax.set_title(f"{row['label']} ({step_label})")
    ax.grid(False)
axes[-1].set_xlabel('X [m]')
fig.colorbar(last_mappable, ax=axes, label='log10 resistivity [ohm m]', shrink=0.9)
plt.show()

## Difference from baseline

这张图用 `log10(rho_case) - log10(rho_baseline)` 看每个正则化相对默认结果改变了哪里。

In [ ]:
baseline_row = next((r for r in available_rows if r['case'] == 'weighted_log_l2_baseline'), available_rows[0])
baseline_out = Path(baseline_row['out_dir'])
baseline_models = load_models(baseline_out)
base = np.log10(np.maximum(baseline_models[:, plot_time_index], np.finfo(float).tiny))

compare_rows = [r for r in available_rows if r['case'] != baseline_row['case']]
if compare_rows:
    diffs = []
    for row in compare_rows:
        models = load_models(Path(row['out_dir']))
        diffs.append(np.log10(np.maximum(models[:, plot_time_index], np.finfo(float).tiny)) - base)
    all_diffs = np.concatenate(diffs)
    lim = float(np.nanpercentile(np.abs(all_diffs), 98))
    lim = max(lim, 1e-6)

    fig, axes = plt.subplots(len(compare_rows), 1, figsize=(11, 2.1 * len(compare_rows)), constrained_layout=True)
    if len(compare_rows) == 1:
        axes = [axes]
    last_mappable = None
    nodes, cells = load_mesh(baseline_out)
    for ax, row, diff in zip(axes, compare_rows, diffs):
        last_mappable = ax.tripcolor(nodes[:, 0], nodes[:, 1], cells, facecolors=diff, shading='flat', cmap='coolwarm', vmin=-lim, vmax=lim)
        ax.set_aspect('equal', adjustable='box')
        ax.set_ylabel('Z [m]')
        ax.set_title(f"{row['label']} - {baseline_row['label']} ({step_label})")
        ax.grid(False)
    axes[-1].set_xlabel('X [m]')
    fig.colorbar(last_mappable, ax=axes, label='Delta log10 resistivity', shrink=0.9)
    plt.show()
else:
    print('Only baseline case is available; no difference plot.')


## Representative cell time series

从网格里选几个代表位置，看不同正则化下随时间的电阻率变化是否更稳定/更符合水文过程。

默认选三个点：浅层、中层、深层附近。你也可以改 `POINTS`。

In [ ]:
def cell_centers(nodes: np.ndarray, cells: np.ndarray) -> np.ndarray:
    return nodes[cells].mean(axis=1)

nodes, cells = load_mesh(ref_out)
centers = cell_centers(nodes, cells)
POINTS = {
    'shallow': (120.0, -8.0),
    'middle': (120.0, -18.0),
    'deep': (120.0, -35.0),
}
point_ids = {}
for name, xy in POINTS.items():
    xy = np.asarray(xy, dtype=float)
    point_ids[name] = int(np.argmin(np.sum((centers - xy) ** 2, axis=1)))
print('Selected cell ids:', point_ids)

fig, axes = plt.subplots(len(point_ids), 1, figsize=(11, 3.0 * len(point_ids)), sharex=True, constrained_layout=True)
if len(point_ids) == 1:
    axes = [axes]

for ax, (point_name, cid) in zip(axes, point_ids.items()):
    for row in available_rows:
        out_dir = Path(row['out_dir'])
        models = load_models(out_dir)
        steps = load_steps(out_dir)
        x = np.arange(models.shape[1]) if steps is None else steps / 24.0
        ax.plot(x, models[cid], lw=1.7, label=row['label'])
    ax.set_ylabel('Resistivity [ohm m]')
    ax.set_title(f"{point_name}: cell {cid}, center=({centers[cid,0]:.1f}, {centers[cid,1]:.1f})")
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('Days')
axes[0].legend(ncol=2, fontsize=9)
plt.show()

## Suggested full-run commands

筛选后，把最好的 1-2 个组合跑全年。例如：

```bash
uv run python examples/2_timelapsedERT_inversion_deepert.py \
  --optimizer gauss_newton_cgls \
  --output-dir ../result/2_timelapsedERT_inversion_deepert_active_time_constraint \
  --spatial-regularization first_order_smoothness \
  --temporal-regularization-type active_time_constraint
```

如果要测试结构先验版本：

```bash
uv run python examples/2_timelapsedERT_inversion_deepert.py \
  --optimizer gauss_newton_cgls \
  --output-dir ../result/2_timelapsedERT_inversion_deepert_structural_prior \
  --spatial-regularization structural_prior \
  --temporal-regularization-type temporal_smoothness
```

如果要单独比较优化算法，可以先保持正则化不变，只改 `--optimizer`，例如 `levenberg_marquardt`, `lbfgs`, `lbfgs_b`, `nonlinear_cg`, `adam`。
